In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('orders (1).csv')
df.head()

,Time,Type,Instrument,Product,Qty.,Avg. price,Status
0,18-12-2021 10:23,SELL,SBI,CNC,1000/1000,525.25,COMPLETE
1,16-12-2021 15:08,BUY,ASHOKLEY,MIS,1000/1000,125.70,COMPLETE
2,16-12-2021 15:08,BUY,SBI,CNC,1000/1000,520.80,COMPLETE
3,16-12-2021 14:13,BUY,TATAMOTORS,MIS,250/250,490.55,COMPLETE
4,16-12-2021 13:54,BUY,ASHOKLEY,MIS,0/1000,127.10,CANCELLED


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Time        14 non-null     object 
 1   Type        14 non-null     object 
 2   Instrument  14 non-null     object 
 3   Product     14 non-null     object 
 4   Qty.        14 non-null     object 
 5   Avg. price  14 non-null     float64
 6   Status      14 non-null     object 
dtypes: float64(1), object(6)
memory usage: 916.0+ bytes


In [4]:
# create a Excel file and upload this to file on sheet1
with pd.ExcelWriter("Kite_Pnl_Report.xlsx", engine='xlsxwriter') as Writer:
    df.to_excel(Writer,
               sheet_name = "Individual Transaction Charges", 
               index = False)
    

In [5]:
df= df[
        (df['Product'] == 'MIS') 
        & 
        (df['Status'] == 'COMPLETE')
]
df

,Time,Type,Instrument,Product,Qty.,Avg. price,Status
1,16-12-2021 15:08,BUY,ASHOKLEY,MIS,1000/1000,125.70,COMPLETE
3,16-12-2021 14:13,BUY,TATAMOTORS,MIS,250/250,490.55,COMPLETE
5,16-12-2021 13:21,SELL,TATAMOTORS,MIS,250/250,492.10,COMPLETE
7,16-12-2021 12:39,SELL,ASHOKLEY,MIS,1000/1000,125.96,COMPLETE
8,16-12-2021 12:29,BUY,ASHOKLEY,MIS,2000/2000,125.70,COMPLETE
9,16-12-2021 11:22,SELL,ASHOKLEY,MIS,2000/2000,125.95,COMPLETE
11,16-12-2021 10:46,SELL,ASHOKLEY,MIS,2000/2000,125.95,COMPLETE
12,16-12-2021 10:24,BUY,ASHOKLEY,MIS,1000/1000,125.60,COMPLETE
13,16-12-2021 10:23,BUY,ASHOKLEY,MIS,1000/1000,125.65,COMPLETE


In [6]:
# extracted number_of_shares_executed
df['Qty.'] = df['Qty.'].str.split('/').str[0].astype(int)

In [7]:
# Turnover = Qty * avg price
df['Turnover'] = df['Qty.'] * df['Avg. price']

In [8]:
# Brokerage = turnover * 0.03%
df['Brokerage'] = df['Turnover'] * 0.0003
df['Brokerage'] = df['Brokerage'].clip(upper=20)

In [9]:
# STT/CTT
# Rate = 0.025%
df['STT/CTT'] = 0
df.loc[df['Type'] == 'SELL', 'STT/CTT'] = df['Turnover']*0.00025

C:\Users\admin\AppData\Local\Temp\ipykernel_31132\951198917.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[30.75625 31.49    62.975   62.975  ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[df['Type'] == 'SELL', 'STT/CTT'] = df['Turnover']*0.00025


In [10]:
# ETC = Exchange Transaction  Charges
# rate = 0.00345%
df['ETC'] = df['Turnover']*0.0000345

In [11]:
# SEBI Charges = Rs. 10 per crore
df['SEBI'] = df['Turnover']*0.000001

In [12]:
# GST = 10% on (Brokerage + ETC + SEBI)
df['GST'] = 0.18 * (df['Brokerage'] + df['ETC'] + df['SEBI'])

In [13]:
# Stamp Charges = aplly on buy side  = 0.003%
df['Stamp'] = 0
df.loc[df['Type'] == 'BUY', 'Stamp'] = df['Turnover']* 0.00003

C:\Users\admin\AppData\Local\Temp\ipykernel_31132\1099679033.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[3.771    3.679125 7.542    3.768    3.7695  ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[df['Type'] == 'BUY', 'Stamp'] = df['Turnover']* 0.00003


In [14]:
# Total Charges = Brokerage + STT + ETC + SEBI + GST + Stamp
df['Total Charges'] = ( df['Brokerage'] + df['STT/CTT'] + df['ETC'] + df['SEBI'] + df['GST'] + df['Stamp'] )

In [15]:
df

,Time,Type,Instrument,Product,Qty.,Avg. price,Status,Turnover,Brokerage,STT/CTT,ETC,SEBI,GST,Stamp,Total Charges
1,16-12-2021 15:08,BUY,ASHOKLEY,MIS,1000,125.70,COMPLETE,125700.0,20.0,0.00000,4.336650,0.125700,4.403223,3.771000,32.636573
3,16-12-2021 14:13,BUY,TATAMOTORS,MIS,250,490.55,COMPLETE,122637.5,20.0,0.00000,4.230994,0.122637,4.383654,3.679125,32.416410
5,16-12-2021 13:21,SELL,TATAMOTORS,MIS,250,492.10,COMPLETE,123025.0,20.0,30.75625,4.244362,0.123025,4.386130,0.000000,59.509767
7,16-12-2021 12:39,SELL,ASHOKLEY,MIS,1000,125.96,COMPLETE,125960.0,20.0,31.49000,4.345620,0.125960,4.404884,0.000000,60.366464
8,16-12-2021 12:29,BUY,ASHOKLEY,MIS,2000,125.70,COMPLETE,251400.0,20.0,0.00000,8.673300,0.251400,5.206446,7.542000,41.673146
9,16-12-2021 11:22,SELL,ASHOKLEY,MIS,2000,125.95,COMPLETE,251900.0,20.0,62.97500,8.690550,0.251900,5.209641,0.000000,97.127091
11,16-12-2021 10:46,SELL,ASHOKLEY,MIS,2000,125.95,COMPLETE,251900.0,20.0,62.97500,8.690550,0.251900,5.209641,0.000000,97.127091
12,16-12-2021 10:24,BUY,ASHOKLEY,MIS,1000,125.60,COMPLETE,125600.0,20.0,0.00000,4.333200,0.125600,4.402584,3.768000,32.629384
13,16-12-2021 10:23,BUY,ASHOKLEY,MIS,1000,125.65,COMPLETE,125650.0,20.0,0.00000,4.334925,0.125650,4.402903,3.769500,32.632979


In [16]:
# Update sheet1
with pd.ExcelWriter("Kite_Pnl_Report.xlsx", engine='xlsxwriter') as Writer:
    df.to_excel(Writer,
               sheet_name = "Individual Transaction Charges", 
               index = False)

In [17]:
df

,Time,Type,Instrument,Product,Qty.,Avg. price,Status,Turnover,Brokerage,STT/CTT,ETC,SEBI,GST,Stamp,Total Charges
1,16-12-2021 15:08,BUY,ASHOKLEY,MIS,1000,125.70,COMPLETE,125700.0,20.0,0.00000,4.336650,0.125700,4.403223,3.771000,32.636573
3,16-12-2021 14:13,BUY,TATAMOTORS,MIS,250,490.55,COMPLETE,122637.5,20.0,0.00000,4.230994,0.122637,4.383654,3.679125,32.416410
5,16-12-2021 13:21,SELL,TATAMOTORS,MIS,250,492.10,COMPLETE,123025.0,20.0,30.75625,4.244362,0.123025,4.386130,0.000000,59.509767
7,16-12-2021 12:39,SELL,ASHOKLEY,MIS,1000,125.96,COMPLETE,125960.0,20.0,31.49000,4.345620,0.125960,4.404884,0.000000,60.366464
8,16-12-2021 12:29,BUY,ASHOKLEY,MIS,2000,125.70,COMPLETE,251400.0,20.0,0.00000,8.673300,0.251400,5.206446,7.542000,41.673146
9,16-12-2021 11:22,SELL,ASHOKLEY,MIS,2000,125.95,COMPLETE,251900.0,20.0,62.97500,8.690550,0.251900,5.209641,0.000000,97.127091
11,16-12-2021 10:46,SELL,ASHOKLEY,MIS,2000,125.95,COMPLETE,251900.0,20.0,62.97500,8.690550,0.251900,5.209641,0.000000,97.127091
12,16-12-2021 10:24,BUY,ASHOKLEY,MIS,1000,125.60,COMPLETE,125600.0,20.0,0.00000,4.333200,0.125600,4.402584,3.768000,32.629384
13,16-12-2021 10:23,BUY,ASHOKLEY,MIS,1000,125.65,COMPLETE,125650.0,20.0,0.00000,4.334925,0.125650,4.402903,3.769500,32.632979


In [18]:
# Sheet 2 - Stockwise & Type wise Summary

In [19]:
# Weighted_value = qty * avg price
df['Weighted_value'] = df['Qty.'] * df['Avg. price']

In [20]:
Stock_summary = df.groupby(['Instrument','Type']).agg({
    'Qty.':'sum',
    'Turnover':'sum',
    'Brokerage':'sum',
    'STT/CTT':'sum',
    'ETC':'sum',
    'SEBI':'sum',
    'GST':'sum',
    'Stamp':'sum',
    'Total Charges':'sum',
    'Weighted_value':'sum'
}).reset_index()

In [21]:
Stock_summary

,Instrument,Type,Qty.,Turnover,Brokerage,STT/CTT,ETC,SEBI,GST,Stamp,Total Charges,Weighted_value
0,ASHOKLEY,BUY,5000,628350.0,80.0,0.00000,21.678075,0.628350,18.415156,18.850500,139.572081,628350.0
1,ASHOKLEY,SELL,5000,629760.0,60.0,157.44000,21.726720,0.629760,14.824166,0.000000,254.620646,629760.0
2,TATAMOTORS,BUY,250,122637.5,20.0,0.00000,4.230994,0.122637,4.383654,3.679125,32.416410,122637.5
3,TATAMOTORS,SELL,250,123025.0,20.0,30.75625,4.244362,0.123025,4.386130,0.000000,59.509767,123025.0


In [22]:
Stock_summary['Avg Price'] = Stock_summary['Weighted_value'] / Stock_summary['Qty.']

In [23]:
Stock_summary = Stock_summary.drop(columns=['Weighted_value'])

In [24]:
Stock_summary = Stock_summary[
[
    'Instrument','Type','Qty.','Avg Price','Turnover','Brokerage','STT/CTT','ETC','SEBI','GST','Stamp','Total Charges'
]]

In [25]:
Stock_summary

,Instrument,Type,Qty.,Avg Price,Turnover,Brokerage,STT/CTT,ETC,SEBI,GST,Stamp,Total Charges
0,ASHOKLEY,BUY,5000,125.670,628350.0,80.0,0.00000,21.678075,0.628350,18.415156,18.850500,139.572081
1,ASHOKLEY,SELL,5000,125.952,629760.0,60.0,157.44000,21.726720,0.629760,14.824166,0.000000,254.620646
2,TATAMOTORS,BUY,250,490.550,122637.5,20.0,0.00000,4.230994,0.122637,4.383654,3.679125,32.416410
3,TATAMOTORS,SELL,250,492.100,123025.0,20.0,30.75625,4.244362,0.123025,4.386130,0.000000,59.509767


In [26]:
# update sheet 2
with pd.ExcelWriter("Kite_Pnl_Report.xlsx") as Writer:
    df.to_excel(Writer,
               sheet_name="Individual Transaction Charges",
               index=False)
    Stock_summary.to_excel(Writer,
                          sheet_name="Stockwise & Type wise Summary",
                          index=False)

In [27]:
# sheet 3  = Stockwise_Summary

In [35]:
# Seperate buy and sell
buy = df[df['Type'] == 'BUY'].groupby('Instrument')['Turnover'].sum()
sell = df[df['Type'] == 'SELL'].groupby('Instrument')['Turnover'].sum()

In [36]:
# Calculate gross_pnl
Gross_Pnl = sell - buy

In [37]:
Gross_Pnl = Gross_Pnl.reset_index()
Gross_Pnl.columns = ['Instrument','gross_pnl']

In [38]:
# calculate total charges per stock
Charges = df.groupby('Instrument')['Total Charges'].sum().reset_index()

In [39]:
Charges.columns = ['Instrument','Total charges']

In [42]:
# merge goss_pnl and charges
summary = pd.merge(Gross_Pnl, Charges, on='Instrument')

In [43]:
summary

,Instrument,gross_pnl,Total charges
0,ASHOKLEY,1410.0,394.192728
1,TATAMOTORS,387.5,91.926177


In [45]:
# calculate Net Pnl
summary['Net Pnl'] = summary['gross_pnl'] - summary['Total charges']

In [47]:
# Calculate % charges on gross pnl
summary['% Charges on Gross Pnl'] = (summary['Total charges'] / summary['gross_pnl'])*100

In [48]:
summary

,Instrument,gross_pnl,Total charges,Net Pnl,% Charges on Gross Pnl
0,ASHOKLEY,1410.0,394.192728,1015.807272,27.956931
1,TATAMOTORS,387.5,91.926177,295.573823,23.722884


In [50]:
# update sheet 3
with pd.ExcelWriter("Kite_Pnl_Report.xlsx") as Writer:
    df.to_excel(Writer,
               sheet_name = "Individual Transaction Charges",
               index=False)
    Stock_summary.to_excel(Writer,
               sheet_name = "Stockwise & Type wise Summary",
               index=False)
    summary.to_excel(Writer,
               sheet_name = "Stockwise Summary",
               index=False)
    